# Backbone N heavy-atom neighbors by SS × exposure (PANAV, 10-seed ensemble)

Per frame, heavy atoms within 6 Å of the backbone N, excluding its own residue: `n_heavy` (all) and `n_polar` (non-carbon), reduced to mean and variance over the trajectory.

In [ ]:
import json
import glob
import os
import re
import time
import numpy as np
import pandas as pd
import mdtraj as mdt
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import mannwhitneyu, wilcoxon, pearsonr, spearmanr
import warnings
warnings.filterwarnings("ignore")


def set_style(base=13, family="Arial"):
    plt.rcParams.update({
        "font.family": family,
        "font.size": base,
        "axes.titlesize": base,
        "axes.labelsize": base,
        "xtick.labelsize": base - 2,
        "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2,
        "svg.fonttype": "none",
    })


set_style()

RES = "../../../data/md_1us"  # 1-us MD trajectories, one frame per ns
PRED = "../../../data/analysis/Fig4/results/predictions_N_seed_ensemble_10seed.json"
GT_PATH = "../../../data/rBMRB/cluster_30p/PANAV/test.json"
REC_PATH = "../../../data/analysis/Fig4/sasa_results/per_residue_records.json"

FIG = "Figs"
CACHE = "../../../data/analysis/Fig4/per_protein_analysis/nheavy_exposure_cache_panav_10seedens"   # own namespace
os.makedirs(CACHE, exist_ok=True)

ENTRIES = ["11139_1", "36329_1", "30706_1", "6032_1", "16603_1", "10267_1", "6726_1", "52187_1", "19743_1", "11202_1",
           "34106_1", "4463_1", "4819_1", "5921_1", "6933_1", "16628_1", "34043_1", "26517_1", "34425_1", "11188_1",
           "19298_1", "25895_1", "52088_1", "16387_1", "5902_1", "6782_1", "10235_1", "19737_1", "17136_1", "34607_1",
           "50270_1", "6114_1", "25968_1", "5842_1", "27687_1", "16016_1", "25448_1", "36327_1", "51126_1", "17723_1",
           "16848_1", "4671_1", "5490_1", "34678_1", "19231_1", "18145_1", "16652_1", "16186_1"]

DROP = 100          # drop first 100 frames (100 ns equilibration at 1 ns/frame)
CUTOFF_NM = 0.8      # 6 Angstrom neighbor cutoff around the backbone N atom
SS_ORDER = ["helix", "sheet", "coil"]
EXPOSURES = ["buried", "exposed"]
COL = {"buried": "#61AFEF", "exposed": "#E06C75"}


def in_gate(x, r):
    """None / Proline exclusion; no 4-sigma BMRB outlier filter (already part of the prediction mask)."""
    return x is not None and r != "P"


ST = json.load(open(PRED))        # seeds, pred, mask, mean, std (10-seed ensemble)
GT = json.load(open(GT_PATH))     # experimental N, sequence -- PANAV test set (entry_ID value == pdb_id, no "bmr" prefix)
REC = json.load(open(REC_PATH))   # resnum, residue_type, ss_class, exposure (no s2 / p_exchange fields)
key_of = {v: k for k, v in GT["entry_ID"].items()}

## Compute per protein (slow, cached)

In [ ]:
_FRAME_RE = re.compile(r"^frame_?(\d+)")


def _frame_idx(p):
    return int(_FRAME_RE.match(os.path.basename(p)).group(1))


CSV_OUT = f"{CACHE}/nheavy_exposure_amides_panav10seedens.csv"
RECOMPUTE = not os.path.exists(CSV_OUT)

if RECOMPUTE:
    rows = []
    for e in ENTRIES:
        t0 = time.time()
        gk = key_of[e]              # PANAV entry_ID value IS the pdb_id, no "bmr" prefix
        seq = GT["sequence"][gk]
        Nv = GT["N"][gk]
        rec = REC[e]
        L = len(seq)

        # ---- chemical shift side (padding-safe, PANAV/10seedens convention) ----
        pred_arr_full = np.array(ST["mean"][e])            # 10-seed ensemble mean
        pred_mask_full = np.array(ST["mask"][e], bool)
        if len(pred_arr_full) < L or len(pred_mask_full) < L:
            print(f"  !! {e}: prediction array shorter than sequence ({len(pred_arr_full)} < {L}), skipping")
            continue
        pred_arr = pred_arr_full[:L].astype(float)          # truncate batch padding, cast off object dtype
        pred_mask = pred_mask_full[:L]

        gate = np.array([in_gate(Nv[i], seq[i]) and pred_mask[i] for i in range(L)])
        t_all = np.array([Nv[i] if gate[i] else np.nan for i in range(L)], dtype=float)
        signed = np.where(gate, pred_arr - t_all, np.nan)
        acerr = np.abs(signed)          # no per-protein offset correction (PANAV referencing already consistent)

        # ---- trajectory ----
        pdbs = sorted(
            (p for p in glob.glob(f"{RES}/{e}/frame*.pdb") if _FRAME_RE.match(os.path.basename(p))),
            key=_frame_idx,
        )
        win = mdt.load(pdbs)[DROP:]
        top = win.top
        F = win.n_frames

        # ---- heavy-atom bookkeeping, once per protein ----
        heavy_idx = np.array([a.index for a in top.atoms if a.element.symbol != "H"])
        is_carbon = np.array([top.atom(idx).element.symbol == "C" for idx in heavy_idx])
        heavy_pos = {idx: k for k, idx in enumerate(heavy_idx)}   # heavy_idx position -> is_carbon lookup

        n_kept, n_skipped = 0, 0
        for res in top.residues:
            i = res.index
            if i >= L or i >= len(rec):
                continue
            ssc = rec[i]["ss_class"]
            exp = rec[i]["exposure"]
            if ssc not in SS_ORDER or exp not in EXPOSURES:
                continue
            n_atoms = [a.index for a in res.atoms if a.name == "N"]
            if not n_atoms:
                continue
            if not gate[i]:
                n_skipped += 1
                continue
            n_kept += 1

            n_idx = n_atoms[0]

            haystack = heavy_idx                                            # the residue itself is not excluded
            haystack_is_carbon = np.array([top.atom(idx).element.symbol == "C" for idx in haystack])

            neighbors_per_frame = mdt.compute_neighbors(
                win, CUTOFF_NM, query_indices=[n_idx], haystack_indices=haystack, periodic=True,
            )

            haystack_pos = {idx: k for k, idx in enumerate(haystack)}
            n_heavy_per_frame = np.empty(F, dtype=float)
            n_polar_per_frame = np.empty(F, dtype=float)
            for f, nb in enumerate(neighbors_per_frame):
                n_heavy_per_frame[f] = len(nb)
                if len(nb):
                    pos = np.array([haystack_pos[idx] for idx in nb])
                    n_polar_per_frame[f] = int((~haystack_is_carbon[pos]).sum())
                else:
                    n_polar_per_frame[f] = 0

            rows.append(dict(
                entry=e, resid=i, resname=res.name, ss=ssc, exposure=exp,
                mean_n_heavy=float(n_heavy_per_frame.mean()) + 1.0, var_n_heavy=float(n_heavy_per_frame.var()),
                mean_n_polar=float(n_polar_per_frame.mean()) + 1.0, var_n_polar=float(n_polar_per_frame.var()),
                err=float(acerr[i]), n_frames=F,
            ))
        print(f"{e:9s} [{time.time() - t0:.0f}s]  kept={n_kept}  skipped(gate)={n_skipped}", flush=True)

    A = pd.DataFrame(rows)
    A.to_csv(CSV_OUT, index=False)
else:
    A = pd.read_csv(CSV_OUT)

print(f"\n{len(A)} amides over {A.entry.nunique()} proteins")
A.head()

## Buried vs exposed inside each secondary structure

In [ ]:
def report(D, tag):
    print("=" * 100)
    print(f"{tag}   (n_amides={len(D)}, n_proteins={D.entry.nunique()})")
    print("=" * 100)
    METRICS = ["mean_n_heavy", "var_n_heavy", "mean_n_polar", "var_n_polar"]

    def _summarize(g):
        out = {"n": len(g), "n_prot": g["entry"].nunique()}
        for m in METRICS:
            out[m] = g[m].mean()
            out[f"{m}_std"] = g[m].std()
        return pd.Series(out)

    summary = D.groupby(["ss", "exposure"]).apply(_summarize)
    print(summary.round(3).to_string())
    print("  (mean_n_* = average heavy-atom neighbor count within 6A of N; "
          "var_n_* = how much that count fluctuates over the trajectory)")
    print("  (*_std = sample std within the group, i.e. across amides -- NOT the SEM of the group mean)")


A = pd.read_csv(CSV_OUT)
A = A[~A.resname.isin(['PRO'])]
summary_all = report(A, "ALL amides")

## Per-entry pooled correlation: n_heavy / n_polar vs |N error|

In [ ]:
MIN_RES = 5
EXPOSURE_ORDER = ["buried", "exposed"]
PREDICTORS = ["mean_n_heavy", "var_n_heavy", "mean_n_polar", "var_n_polar"]


def pool_by_entry(sub, xcol):
    """One row per entry: mean predictor + median err (err is heavy-tailed, hence the median)."""
    d = sub[[xcol, "err", "entry"]].dropna()
    g = d.groupby("entry").agg(x=(xcol, "mean"), err=("err", "median"), n=("err", "size"))
    return g[g.n >= MIN_RES]


def corr(g):
    if len(g) < 5:
        return dict(n_prot=len(g), rho=np.nan, rp=np.nan, r=np.nan, pp=np.nan)
    rho, rp = spearmanr(g.x, g.err)
    r, pp = pearsonr(g.x, g.err)
    return dict(n_prot=len(g), rho=rho, rp=rp, r=r, pp=pp)


for xcol in PREDICTORS:
    print("=" * 88)
    print(f"per-entry pooled  {xcol}  vs  err     (one point per protein: mean {xcol} vs median err)")
    print("=" * 88)
    print(f"{'stratum':22s} {'n_prot':>7s} {'med_res':>8s} | {'Spearman':>9s} {'p':>10s} | "
          f"{'Pearson':>8s} {'p':>10s}")
    print("-" * 88)
    for exp in EXPOSURE_ORDER:
        for ss in SS_ORDER:
            g = pool_by_entry(A[(A.exposure == exp) & (A.ss == ss)], xcol)
            c = corr(g)
            med_res = int(g.n.median()) if len(g) else 0
            print(f"{exp + '/' + ss:22s} {c['n_prot']:7d} {med_res:8d} | "
                  f"{c['rho']:+9.3f} {c['rp']:10.2e} | {c['r']:+8.3f} {c['pp']:10.2e}")
        g = pool_by_entry(A[A.exposure == exp], xcol)
        c = corr(g)
        print(f"{exp + ' (all ss)':22s} {c['n_prot']:7d} {int(g.n.median()):8d} | "
              f"{c['rho']:+9.3f} {c['rp']:10.2e} | {c['r']:+8.3f} {c['pp']:10.2e}")
        print("-" * 88)
    g = pool_by_entry(A, xcol)
    c = corr(g)
    print(f"{'ALL':22s} {c['n_prot']:7d} {int(g.n.median()):8d} | "
          f"{c['rho']:+9.3f} {c['rp']:10.2e} | {c['r']:+8.3f} {c['pp']:10.2e}")
    print()

## Per-entry scatter plots: n_heavy / n_polar vs median |N error|

In [ ]:
def set_style(base=13, family="Arial"):
    plt.rcParams.update({
        "font.family": family,
        "font.size": base,
        "axes.titlesize": base,
        "axes.labelsize": base,
        "xtick.labelsize": base - 2,
        "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2,
        "svg.fonttype": "none",
    })


set_style(base=16)
BLUE, RED, GREEN, GREY, PURPLE = "#61AFEF", "#E06C75", "#98C379", "#999999", "#C678DD"
PREDICTORS = ["mean_n_polar", "var_n_polar", "mean_n_heavy", "var_n_heavy"]

def style_axes(ax):
    # ax.grid(alpha=.3, ls="--")
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)

def save(fig, name):
    fig.savefig(f"{FIG}/{name}.svg", bbox_inches="tight")
    fig.savefig(f"{FIG}/{name}.png", dpi=200, bbox_inches="tight")
    print("saved", name)

def scatter(exp, ss, xcol, fname=None):
    sub = A if ss is None else A[(A.exposure == exp) & (A.ss == ss)]
    g = pool_by_entry(sub, xcol)
    c = corr(g)
    fig, ax = plt.subplots(figsize=(5.4, 5.0))
    ax.scatter(g.x, g.err, s=90, alpha=.7, color=BLUE, edgecolor="none")
    # for entry, r in g.iterrows():
    #     ax.annotate(entry.split("_")[0], (r.x, r.err), fontsize=8,
    #                 xytext=(3, 3), textcoords="offset points")
    if len(g) >= 2:
        b1, b0 = np.polyfit(g.x, g.err, 1)
        xs = np.linspace(g.x.min(), g.x.max(), 20)
        ax.plot(xs, b1 * xs + b0, color=RED, lw=2, label="fit")
        ax.legend(frameon=False)
    if xcol == "var_n_heavy":
        xcol = "var_pack_N8"
    ax.set_xlabel(f"mean {xcol}  ({exp}/{ss})")
    ax.set_ylabel("median |N error| (ppm)")
    ax.set_title(f"Pearson={c['r']:.2f}  Spearman={c['rho']:.2f}\nNumber of proteins = {c['n_prot']}")
    style_axes(ax)
    plt.tight_layout()
    if fname:
        save(fig, fname)
    plt.show()

EXPOSURE_ORDER = ["exposed"]
SS_ORDER = ["helix", "coil", "sheet"]

for exp in EXPOSURE_ORDER:
    for ss in SS_ORDER:
        for xcol in PREDICTORS:
            scatter(exp, ss, xcol, fname=f"perentry_{xcol}_vs_err_{exp}_{ss}_panav10seedens")

In [ ]:
# ================= cross-model robustness, exposed coil: per-protein N-centred heavy-atom crowding vs |N error| =================
# Self-contained. For each model, |error| is recomputed from its own 10-seed-ensemble predictions with
# the same gate as the cached CSV (Pro / None / prediction mask); the MD crowding metrics come from the CSV
# and are model-independent (the CSV's own `err` column is NOT used -- it predates the 2026-09-25 predictions).
# Per protein (same as pool_by_entry): mean predictor vs median |error|, proteins with < MIN_RES residues in
# the stratum dropped. Bar = Spearman rho, whisker = 95% CI from a protein bootstrap, text = Spearman p.
# One figure per predictor in XCOLS, shared y-range.
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, to_hex
from scipy.stats import spearmanr

FIG4 = "../../../data/analysis/Fig4"
FIG = "Figs"
NHEAVY_CSV = f"{FIG4}/per_protein_analysis/nheavy_exposure_cache_panav_10seedens/nheavy_exposure_amides_panav10seedens.csv"
GT = "../../../data/rBMRB/cluster_30p/PANAV/test.json"
MODELS = {                      # label -> prediction-file suffix; order = bar order
    "SLAE-CS":       "",
    "ESM3\n(str + seq)":          "_esm3",
    "ESM3\n(seq only)": "_esm3_seqonly",
    "MACE-OFF23":          "_mace",
    "ProteinMPNN\n(finetuned)":       "_mpnn_ft",
}
DARK_BLUE, LIGHT_BLUE = "#3F729B", "#A9CCE3"
_grad = LinearSegmentedColormap.from_list("model_blue", [DARK_BLUE, LIGHT_BLUE])
MODEL_C = {"SLAE-CS": "#3F729B", "ESM3\n(str + seq)": "#A9CCE3", "ESM3\n(seq only)": "#A9CCE3",
           "MACE-OFF23": "#A9CCE3", "ProteinMPNN\n(finetuned)": "#A9CCE3", "ProteinMPNN\n(frozen)": "#6E6E6E"}
STRATUM = ("exposed", "coil")
XCOLS = {                       # predictor -> title label; add "mean_n_heavy" / "*_n_polar" here if needed
    "var_n_heavy":  "variance of pack_N8",
}
MIN_RES = 5
B_BOOT = 2000


def set_style(base=13, family="Arial"):
    plt.rcParams.update({
        "font.family": family, "font.size": base,
        "axes.titlesize": base, "axes.labelsize": base,
        "xtick.labelsize": base - 3, "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2, "svg.fonttype": "none",
    })


# ---------- per-residue table: MD crowding metrics from the cached CSV (already gated, Pro excluded) ----------
A = pd.read_csv(NHEAVY_CSV)
A = A[A.resname != "PRO"].copy()
A["resnum"] = A["resid"] + 1                  # mdtraj 0-based residue index -> 1-based resnum
A["gated"] = A["err"].notna()                 # gate membership only; the stale err values are dropped below
A = A.drop(columns="err")

# ---------- |error| per model, same gate recipe (Pro / None / prediction mask) ----------
gt = json.load(open(GT))
inv = {v: k for k, v in gt["entry_ID"].items()}
for label, tag in MODELS.items():
    pred = json.load(open(f"{FIG4}/results/predictions_N_seed_ensemble_10seed{tag}.json"))
    err = {}
    for e in A.entry.unique():
        seq, shifts = gt["sequence"][inv[e]], gt["N"][inv[e]]
        n = len(seq)
        pa = np.array(pred["mean"][e])[:n]
        pm = np.array(pred["mask"][e], dtype=bool)[:n]
        for i in range(n):
            if seq[i] != "P" and shifts[i] is not None and pm[i]:
                err[(e, i + 1)] = abs(float(pa[i]) - float(shifts[i]))
    A[label] = [err.get((e, r), np.nan) if g else np.nan for e, r, g in zip(A.entry, A.resnum, A.gated)]
print("residues with |error| per model:", {m: int(A[m].notna().sum()) for m in MODELS})

# ---------- per-protein Spearman + protein bootstrap CI (exposed coil) ----------
rng = np.random.default_rng(0)
exp, ss = STRATUM
sub = A[(A.exposure == exp) & (A.ss == ss)]
rows = []
for xcol in XCOLS:
    for m in MODELS:
        g = (sub[["entry", xcol, m]].dropna()
             .groupby("entry").agg(x=(xcol, "mean"), err=(m, "median"), n=(m, "size")))
        g = g[g.n >= MIN_RES]
        rho, p = spearmanr(g.x, g.err)
        x, y = g.x.to_numpy(), g.err.to_numpy()
        boot = [spearmanr(x[i], y[i])[0] for i in (rng.integers(0, len(g), len(g)) for _ in range(B_BOOT))]
        lo, hi = np.nanpercentile(boot, [2.5, 97.5])
        rows.append(dict(predictor=xcol, model=m, n_prot=len(g), med_res=int(g.n.median()),
                         rho=rho, ci_lo=lo, ci_hi=hi, p=p))
RR = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(f"{exp} {ss}: per protein, mean predictor vs median |error|")
print(RR.round(3).to_string(index=False))


In [ ]:

# ---------- one bar chart per predictor, one bar per model, shared y-range ----------
set_style(base=16)
Y_LO = min(RR.ci_lo.min(), 0) - 0.16
Y_HI = max(RR.ci_hi.max(), 0) + 0.08
xs = np.arange(len(MODELS))
for xcol, xlabel in XCOLS.items():
    R = RR[RR.predictor == xcol].set_index("model").loc[list(MODELS)]
    fig, ax = plt.subplots(figsize=(7.5, 5.6))
    ax.bar(xs, R.rho, 0.7, color=[MODEL_C[m] for m in MODELS], zorder=2)
    ax.errorbar(xs, R.rho, yerr=[R.rho - R.ci_lo, R.ci_hi - R.rho], fmt="none",
                ecolor="#444", elinewidth=1.2, capsize=4, zorder=3)
    for x, (m, r) in zip(xs, R.iterrows()):
        sig = r.p < 0.05
        ax.text(x, Y_LO + 0.02, f"p={r.p:.2g}" if r.p >= 1e-3 else f"p={r.p:.0e}", ha="center", va="bottom",
                fontsize=14, fontweight="bold" if sig else "normal", color="k" if sig else "#777", font="Arial")
    ax.axhline(0, color="k", lw=1)
    ax.set_xticks(xs)
    ax.set_xticklabels(list(MODELS))
    ax.set_ylim(-1, 0.35)
    ax.set_ylabel("Spearman ρ (per protein)")
    ax.set_title(f"Exposed coil: {xlabel} vs median |error|",
                 fontsize=14, font="Arial")
    for s_ in ("top", "right"):
        ax.spines[s_].set_visible(False)
    plt.tight_layout()
    name = f"nheavy_{xcol.replace('_', '')}_perprotein_rho_models_exposedcoil_panav10seedens"
    fig.savefig(f"{FIG}/{name}.svg", bbox_inches="tight")
    fig.savefig(f"{FIG}/{name}.png", dpi=300, bbox_inches="tight")
    print("saved", name)
    plt.show()
